# LangGraph Minimal Workflow Test
This notebook validates a minimal end-to-end LangGraph workflow, from install to assertions.

## 1. Install and Verify LangGraph

In [1]:
# Install LangGraph and verify version
!pip -q install --disable-pip-version-check langgraph

from importlib import metadata as md

print("langgraph version:", md.version("langgraph"))

langgraph version: 1.1.10


## 2. Import Dependencies and Configure Environment

In [2]:
# Import core LangGraph utilities and set environment flags
import os
from typing import TypedDict

from langgraph.graph import StateGraph, START, END

os.environ["LANGGRAPH_TEST"] = "1"
print("LANGGRAPH_TEST:", os.environ.get("LANGGRAPH_TEST"))

LANGGRAPH_TEST: 1


## 3. Define a Minimal State and Nodes

In [3]:
# Define a minimal state and simple node functions
class CounterState(TypedDict):
    value: int
    route: str


def increment(state: CounterState) -> CounterState:
    return {**state, "value": state["value"] + 1, "route": "check"}


def decide(state: CounterState) -> str:
    return "end" if state["value"] >= 2 else "increment"

## 4. Build the StateGraph and Edges

In [4]:
# Build the graph with a simple conditional route
builder = StateGraph(CounterState)

builder.add_node("increment", increment)

builder.add_edge(START, "increment")
builder.add_conditional_edges(
    "increment",
    decide,
    {
        "increment": "increment",
        "end": END,
    },
)

app = builder.compile()
print("Graph compiled")

Graph compiled


## 5. Invoke the Graph with Test Inputs

In [5]:
# Run the graph with sample input
initial_state: CounterState = {"value": 0, "route": "check"}
final_state = app.invoke(initial_state)

print("Initial state:", initial_state)
print("Final state:", final_state)

Initial state: {'value': 0, 'route': 'check'}
Final state: {'value': 2, 'route': 'check'}


## 6. Assert Expected Outputs

In [6]:
# Validate the final state
assert final_state["value"] == 2, f"Expected value=2, got {final_state['value']}"
assert isinstance(final_state, dict)
print("LangGraph test complete")

LangGraph test complete


In [7]:
print("RAW CF_ACCOUNT_ID =", repr(os.environ.get("CF_ACCOUNT_ID")))
print("LEN =", len(os.environ.get("CF_ACCOUNT_ID", "")))

RAW CF_ACCOUNT_ID = None
LEN = 0


## 7. Run the Team 04 Workflow

In [ ]:
# Run the real Team 04 workflow with user input

import importlib
import os
import re
import time
from pathlib import Path

import central_reasoning_node as crn
import design_config
import design_workflow_graph as dwg

from dotenv import dotenv_values, load_dotenv
from mcp_client import McpClient

# Correct .env path
env_path = Path(r"D:\3rd sem\STUDIO\AIA26_Studio\team_04\PY\.env")

# Check file exists
if not env_path.exists():
    raise FileNotFoundError(f".env not found at: {env_path}")

# Load .env into environment
load_dotenv(dotenv_path=env_path, override=True)

# Read values
env_values = dotenv_values(env_path)
for key, value in env_values.items():
    if value is not None:
        os.environ[key] = value

importlib.reload(design_config)
importlib.reload(crn)
importlib.reload(dwg)

settings = design_config.load_design_settings()

print("ENV LOADED SUCCESSFULLY")
print("Provider =", settings.llm_provider)
print("Model =", settings.llm_model)
print("Base URL =", settings.base_url)

# Cloudflare needs a real account id in the endpoint path.
if settings.llm_provider == "cloudflare":
    account_id = os.environ.get("CF_ACCOUNT_ID", "")
    if not re.fullmatch(r"[0-9a-fA-F]{32}|[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}", account_id):
        raise ValueError(
            'Invalid CF_ACCOUNT_ID. Cloudflare AI requires a real account id in the '
            '/client/v4/accounts/<account_id>/ai/v1 endpoint. '
            'Replace the current placeholder value in .env with your actual Cloudflare account id.'
        )

# USER INPUT — Use simpler prompt for faster testing
prompt = "Generate a procedural site boundary with a total site area of 10,000 square meters, having 5 site edges, oriented at a 45-degree north angle."
#prompt = "Generate a usable site area within the existing site boundary and populate the site with 10 trees having a tree radius of 3 meters. Distribute vegetation evenly while maintaining clearance from the boundary edges and preserving accessible open space. Calculate and display the resulting usable site area after vegetation allocation."

print(f"\n[TIMING] Starting workflow...\n")
mcp_client = McpClient(
    settings.mcp_endpoint,
    settings.request_timeout_seconds,
)

try:
    print("[1/4] Initializing MCP client...")
    init_start = time.perf_counter()
    mcp_client.initialize()
    tools = mcp_client.list_tools()
    print(f"      ✓ Done in {time.perf_counter() - init_start:.2f}s ({len(tools)} tools found)\n")

    print("[2/4] Running LangGraph workflow...")
    workflow_start = time.perf_counter()

    response = dwg.run_design_workflow(
        user_prompt=prompt,
        tools=tools,
        mcp_client=mcp_client,
        api_key=settings.api_key,
        base_url=settings.base_url,
        llm_model=settings.llm_model,
        debug_graph=False,  # Set to False to skip debug output (faster)
        timeout_seconds=settings.request_timeout_seconds,
        max_iterations=1,  # REDUCED from 2 to 1 (faster but less accurate)
    )
    
    workflow_elapsed = time.perf_counter() - workflow_start
    print(f"      ✓ Done in {workflow_elapsed:.2f}s\n")

    response_text = response if isinstance(response, str) else str(response)

    print("[3/4] Response Summary:")
    print(response_text[:1500])
    if len(response_text) > 1500:
        print(f"      ... ({len(response_text) - 1500} more characters)\n")

    print("[4/4] Timing Summary:")
    total_elapsed = time.perf_counter() - workflow_start
    print(f"      • Response length: {len(response_text)} characters")
    print(f"      • Total time: {workflow_elapsed:.2f}s")

finally:
    mcp_client.close()


ENV LOADED SUCCESSFULLY
Provider = cloudflare
Model = @cf/qwen/qwen3-30b-a3b-fp8
Base URL = https://api.cloudflare.com/client/v4/accounts/7f57ff835734991ccdf3e1799878d39a/ai/v1
[workflow] Initializing design workflow
[workflow] Model: @cf/qwen/qwen3-30b-a3b-fp8
[workflow] Max iterations: 2
[workflow] Graph compiled
[workflow][central_reason] Enter node
[workflow][central_reason] Calling LLM


In [ ]:
import json
from design_config import load_design_settings
from mcp_client import McpClient

settings = load_design_settings()
client = McpClient(settings.mcp_endpoint, settings.request_timeout_seconds)
try:
    client.initialize()
    tools = client.list_tools()
    print([tool.get("name") for tool in tools])
    print(json.dumps(tools[:3], indent=2))
finally:
    client.close()

In [ ]:
settings = load_design_settings()
client = McpClient(settings.mcp_endpoint, settings.request_timeout_seconds)
try:
    client.initialize()
    output = client.call_tool(
        "site_boundary_generation",
        {
            "Site area": 10000,
            "Number of edges of sites": 5,
            "Site angle": 45,
        },
    )
    print(output)
finally:
    client.close()